# MERA Quant Bench — деградация метрик MERA по квантам GGUF

Оценивает **все кванты одной модели** (репозиторий `{PUBLISHER}/{MODEL}-GGUF`) на **локально
проверяемых** задачах MERA через предсобранный `llama-server` (llama.cpp v0.4.1 + PR #27537,
echo+logprobs) и форк lm-evaluation-harness от MERA (бэкенд `local-completions`).

**Как использовать:** GPU-рантайм (T4/L4/A100) → секрет `HF_TOKEN` → Runtime → Run all.
Сначала прогоните пресет `smoke` (Блок 3). Полный прогон — несколько сессий: чекпоинт на
Google Drive возобновляет автоматом.

Схема блоков: 0 диагностика · 1 токен · 2 Drive · 3 конфигурация · 4 MERA · 5 llama.cpp ·
6 функции · 7 цикл · 8 агрегация · 9 финал · 10 пересборка из логов.


In [ ]:
# @title БЛОК 0: ДИАГНОСТИКА ОКРУЖЕНИЯ
import os, shutil, subprocess, time, json, gc
from pathlib import Path
from datetime import datetime
import torch

print(f"🖥️ Диагностика: {datetime.now():%Y-%m-%d %H:%M:%S}")
if not torch.cuda.is_available():
    raise RuntimeError("❌ GPU не найден. Среда выполнения → Сменить среду выполнения → T4/L4/A100.")

props = torch.cuda.get_device_properties(0)
print(f"  GPU: {props.name} ({props.total_memory / 1e9:.1f} ГБ VRAM, CC {props.major}.{props.minor})")

isa = subprocess.run("grep -o -m1 avx512f /proc/cpuinfo", shell=True,
                     capture_output=True, text=True).stdout.strip()
print(f"  CPU avx512f: {'✅' if isa else '❌ нет'} (сборка native требует AVX-512)")
if not isa:
    raise RuntimeError("❌ CPU без AVX-512: native-сборка llama.cpp не запустится. Нужен GPU-рантайм на Xeon (T4/L4/A100).")

ram = {}
with open("/proc/meminfo") as f:
    for line in f:
        if ":" in line:
            k, v = line.split(":", 1)
            ram[k] = int(v.split()[0])
print(f"  RAM: {ram.get('MemTotal', 0) / 1e6:.1f} ГБ (доступно {ram.get('MemAvailable', 0) / 1e6:.1f} ГБ)")
total, used, free = shutil.disk_usage("/")
print(f"  Диск: свободно {free / 1e9:.1f} ГБ")
print("✅ Блок 0 завершён.")


In [ ]:
# @title БЛОК 1: ТОКЕН HUGGING FACE
from google.colab import userdata
from huggingface_hub import HfApi

hf_token = userdata.get("HF_TOKEN")
if not hf_token or not str(hf_token).strip():
    print("⚠️ Токен HF_TOKEN не найден в секретах Colab. Модели из приватных репо будут недоступны.")
    os.environ["HF_TOKEN"] = ""
else:
    try:
        user = HfApi().whoami(token=hf_token)
        print(f"✅ Токен проверен. Добро пожаловать, {user.get('name', 'пользователь')}!")
        os.environ["HF_TOKEN"] = str(hf_token).strip()
    except Exception as e:
        print(f"❌ Токен недействителен: {e}")
        os.environ["HF_TOKEN"] = ""


In [ ]:
# @title БЛОК 2: GOOGLE DRIVE И СВОБОДНОЕ МЕСТО
from google.colab import drive

print("💾 Монтирование Google Drive...")
drive.mount("/content/drive", force_remount=False)
DRIVE_ROOT = "/content/drive"

total, used, free = shutil.disk_usage(DRIVE_ROOT)
print(f"💾 Свободно на Google Диске: {free / 1e9:.2f} ГБ")
if free / 1e9 < 2.0:
    print("⚠️ ВНИМАНИЕ: на Диске меньше 2 ГБ — чекпоинт и сырые логи могут не сохраниться!")


In [ ]:
# @title БЛОК 3: КОНФИГУРАЦИЯ (менять параметры здесь)
import torch

# ---------- Модель (один прогон = одна модель) ----------
PUBLISHER = "unsloth"
BASE_MODEL_NAME = "Qwen3.5-9B"
REPO_ID = f"{PUBLISHER}/{BASE_MODEL_NAME}-GGUF"

# ---------- Системный промпт (опционально) ----------
SYSTEM_PROMPT = (
    "Ты — исполнитель заданий. Выполняй задание буквально и полностью. "
    "Выводи только то, что прямо запрошено заданием, и ничего сверх него: "
    "без приветствий, пояснений, рассуждений и итоговых обобщений. "
    "Если задание требует завершить фрагмент программного кода — выведи только "
    "валидный код на Python, непрерывно продолжающий предложенный фрагмент: "
    "без markdown-разметки (в том числе без тройных обратных кавычек), без текста "
    "до или после кода и без описания решения; заверши вывод непосредственно "
    "после последней строки кода. Если задание требует указать вариант ответа — "
    "укажи только его обозначение. Если задание требует краткого ответа — "
    "выведи только его значение."
)
PRESET = "open_optimum"

# ============================================================
# ВЫБОР АРХИТЕКТУРНОГО ПУТИ (БЭКЕНДА)
# ============================================================
# "hf"                 — Путь A: прямая загрузка GGUF через transformers
#                        (полное соответствие run_benchmark.sh, сервер не нужен)
# "gguf"               — Путь B: нативный бэкенд lm_eval для llama-server
#                        (teacher forcing через logit_bias, автопиннинг слотов)
# "local-completions"  — Путь C: кастомный путь с PR #27537
#                        (echo+logprobs, текущий проверенный вариант)
# ============================================================
BACKEND = "local-completions"  # "hf" | "gguf" | "local-completions"

assert BACKEND in ("hf", "gguf", "local-completions"), \
    f"Неизвестный бэкенд {BACKEND}. Допустимые: hf, gguf, local-completions"

# ============================================================
# ПОЛНЫЙ РЕЕСТР ЗАДАЧ MERA v2.0 (23 задачи)
# ============================================================
TASK_INFO = {
    # ─── Алгоритмы ────────────────────────────────────────────────
    "bps":          {"type": "mc",  "metric": "acc",         "fewshot": 1, "open": True,
                     "desc": "Алгоритмы: сбалансированные скобочные последовательности (бинарная классификация)"},
    "lcs":          {"type": "mc",  "metric": "acc",         "fewshot": 1, "open": False,
                     "desc": "Алгоритмы: длина наибольшей общей подпоследовательности (многоклассовая)"},

    # ─── Знания о мире ────────────────────────────────────────────
    "chegeka":      {"type": "gen", "metric": "exact_match", "fewshot": 1, "open": False,
                     "desc": "Знания о мире: вопросы из ЧГК, Брейн-ринг (открытый ответ, F1 / Exact match)"},
    "ruopenbookqa": {"type": "mc",  "metric": "acc",         "fewshot": 1, "open": False,
                     "desc": "Знания о мире: естественные науки (выбор из 4 вариантов)"},
    "ruworldtree":  {"type": "mc",  "metric": "acc",         "fewshot": 1, "open": False,
                     "desc": "Знания о мире: естественные науки, школа (выбор из 4 вариантов)"},

    # ─── Кодинг ───────────────────────────────────────────────────
    "rucodeeval":   {"type": "gen", "metric": "pass@1",      "fewshot": 0, "open": False,
                     "desc": "Кодинг: генерация кода на Python для алгоритмических задач (CodeEval)"},
    "ruhumaneval":  {"type": "gen", "metric": "pass@1",      "fewshot": 0, "open": True,
                     "desc": "Кодинг: генерация кода на Python для алгоритмических задач (HumanEval)"},

    # ─── Математика ───────────────────────────────────────────────
    "rumultiar":    {"type": "gen", "metric": "exact_match", "fewshot": 1, "open": False,
                     "desc": "Математика: многоступенчатая арифметика со скобками (открытый ответ)"},
    "simplear":     {"type": "gen", "metric": "exact_match", "fewshot": 2, "open": True,
                     "desc": "Математика: сложение n-значных чисел (открытый ответ)"},

    # ─── Математика и логика ──────────────────────────────────────
    "mathlogicqa":  {"type": "mc",  "metric": "acc",         "fewshot": 1, "open": False,
                     "desc": "Математика и логика: задачи на естественном языке (выбор из 4)"},
    "rumodar":      {"type": "gen", "metric": "exact_match", "fewshot": 0, "open": False,
                     "desc": "Математика и логика: арифметика по аналогии (5 примеров + 1)"},

    # ─── Ризонинг ─────────────────────────────────────────────────
    "mamuramu":     {"type": "mc",  "metric": "acc",         "fewshot": 1, "open": False,
                     "desc": "Ризонинг: экспертные задачи на понимание текста (выбор из 4)"},
    "multiq":       {"type": "gen", "metric": "exact_match", "fewshot": 0, "open": False,
                     "desc": "Ризонинг: ответы на вопросы по двум текстам (открытый ответ, F1 / Exact match)"},
    "rummlu":       {"type": "mc",  "metric": "acc",         "fewshot": 1, "open": True,
                     "desc": "Ризонинг: экспертные задачи MMLU, 57 доменов (выбор из 4)"},
    "rwsd":         {"type": "mc",  "metric": "acc",         "fewshot": 1, "open": False,
                     "desc": "Ризонинг: разрешение анафоры, местоимения (Да/Нет)"},
    "use":          {"type": "gen", "metric": "grade_norm",  "fewshot": 1, "open": False,
                     "desc": "Ризонинг: задачи ЕГЭ по русскому языку (открытый ответ, Grade norm)"},

    # ─── Common Sense / NLI ───────────────────────────────────────
    "parus":        {"type": "mc",  "metric": "acc",         "fewshot": 1, "open": False,
                     "desc": "Common Sense: логические связи, причина/следствие (выбор из 2)"},
    "rcb":          {"type": "mc",  "metric": "acc",         "fewshot": 1, "open": False,
                     "desc": "NLI: согласованность, противоречие, независимость (выбор из 3)"},

    # ─── Ризонинг, Диалоговый контекст, Память ────────────────────
    "rutie":        {"type": "mc",  "metric": "acc",         "fewshot": 1, "open": False,
                     "desc": "Эмуляция теста Тьюринга: выбор адекватного ответа в диалоге (выбор из 2)"},

    # ─── Этика ────────────────────────────────────────────────────
    "rudetox":      {"type": "gen", "metric": "j",           "fewshot": 1, "open": True,
                     "desc": "Этика: детоксикация текста с сохранением стиля (открытый ответ, J-метрика)"},
    "ruethics":     {"type": "mc",  "metric": "mcc",         "fewshot": 0, "open": True,
                     "desc": "Этика: 5 измерений оценки поступка (15 метрик MCC)"},
    "ruhatespeech": {"type": "mc",  "metric": "acc",         "fewshot": 1, "open": True,
                     "desc": "Этика: выявление агрессивно окрашенных и токсичных ответов (выбор из 2)"},
    "ruhhh":        {"type": "mc",  "metric": "acc",         "fewshot": 0, "open": True,
                     "desc": "Этика: Helpful / Honest / Harmless (выбор из 2)"},
}

# ---------- Производные списки ----------
_OPEN_TASKS = sorted([t for t, v in TASK_INFO.items() if v["open"]])
_MC_TASKS   = sorted([t for t, v in TASK_INFO.items() if v["type"] == "mc"])
_GEN_TASKS  = sorted([t for t, v in TASK_INFO.items() if v["type"] == "gen"])

PRESETS = {
    "smoke": {
        "title": "Быстрая проверка пайплайна",
        "tasks": ["bps", "simplear"], "limit": 20,
        "hint": "~15 мин/квант. Проверяет ОБЕ механики: echo+logprobs (bps) и генерацию (simplear).",
    },
    "choice": {
        "title": "Выбор ответа — знания и этика (loglikelihood)",
        "tasks": ["bps", "rummlu", "ruethics", "ruhatespeech", "ruhhh"], "limit": 100,
        "hint": "Чистый loglikelihood-путь: главный тест echo+logprobs (PR #27537).",
    },
    "generation": {
        "title": "Генерация ответов (арифметика и код)",
        "tasks": ["simplear", "ruhumaneval", "rucodeeval"], "limit": 100,
        "hint": "Свободная генерация ответов, метрика exact_match / pass@1.",
    },
    "all": {
        "title": "Все локально оцениваемые задачи",
        "tasks": list(TASK_INFO), "limit": 100,
        "hint": "Полный прогон: несколько сессий, чекпоинт возобновит автоматом.",
    },
    "choice_official": {
        "title": "Выбор ответа (Официальный набор, 14 задач)",
        "tasks": _MC_TASKS, "limit": 100,
        "hint": "Все 14 задач классификации (loglikelihood) из официального run_benchmark.sh.",
    },
    "generation_official": {
        "title": "Генерация (Официальный набор, 9 задач)",
        "tasks": _GEN_TASKS, "limit": 100,
        "hint": "Все 9 генеративных задач из официального run_benchmark.sh (GENERATIVE_TASKS).",
    },
    "all_official": {
        "title": "Все задачи (Официальный набор, 23 задачи)",
        "tasks": list(TASK_INFO), "limit": None,
        "hint": "Полный прогон всех 23 задач MERA без лимита.",
    },
    "open": {
        "title": "Открытые датасеты (8 диагностических, локальная проверка)",
        "tasks": _OPEN_TASKS, "limit": 100,
        "hint": "8 диагностических задач с открытыми ответами.",
    },
    "open_optimum": {
        "title": "Открытые датасеты: оптимальный набор (8 задач, максимальное покрытие)",
        "tasks": [
            "simplear",
            "rudetox",
            "bps",
            "rummlu",
            "ruhhh",
            "ruethics",
            "ruhumaneval",
            "ruhatespeech",
        ],
        "limit": 50,
        "hint": (
            "Все открытые задачи в порядке увеличения времени оценки. "
            "Сохраняет полный охват открытых областей: математика, алгоритмы, "
            "кодинг, ризонинг и этика. Для безлимитного прогона замените limit на None."
        ),
    },

    # Быстрый скрининг открытых задач
    "open_screen": {
        "title": "Открытые датасеты: быстрый скрининг (2 задачи)",
        "tasks": [
            "bps",
            "simplear",
        ],
        "limit": None,
        "hint": (
            "Быстрый первичный отбор квантов. "
            "Базовое покрытие: алгоритмы (bps) и математика (simplear)"
        ),
    },

    # Баланс скорости и полноты
    "open_core": {
        "title": "Открытые датасеты: баланс скорости и полноты (6 задач)",
        "tasks": [
            "simplear",
            "rudetox",
            "bps",
            "rummlu",
            "ruhhh",
            "ruhumaneval",
        ],
        "limit": 100,
        "hint": (
            "Сокращенный набор с усиленным этическим блоком за счет ruhhh. "
            "Сохраняет кодинг и ризонинг, но быстрее полного open_optimum."
        ),
    },
}

assert PRESET in PRESETS, f"Неизвестный пресет {PRESET}: {list(PRESETS)}"
TASKS = PRESETS[PRESET]["tasks"]
LIMIT = PRESETS[PRESET]["limit"]

_unknown = [t for t in TASKS if t not in TASK_INFO]
assert not _unknown, f"Неизвестные задачи в пресете: {_unknown}"
if LIMIT is not None:
    assert isinstance(LIMIT, int) and LIMIT > 0, "LIMIT должен быть положительным целым числом"

# ---------- Параметры прогона ----------
SEED = 1234
SERVER_PORT = 8000
MAX_VRAM_GB = None
FORCE_RERUN = False

if MAX_VRAM_GB is None:
    MAX_VRAM_GB = round(torch.cuda.get_device_properties(0).total_memory / 1e9 - 2.0, 1)

# ---------- Сборка llama.cpp ----------
DRIVE = Path(DRIVE_ROOT) / "MyDrive"
LLAMA_CPP_ARCHIVE = str(DRIVE / "llama.cpp_v0.4.1-pr27537-version" / "native" / "gpu_all.tar.gz")
LLAMA_CPP_MANIFEST = str(DRIVE / "llama.cpp_v0.4.1-pr27537-version" / "manifest.json")

# ---------- Пути вывода ----------
SAVE_DIR = DRIVE / f"{PUBLISHER}_{BASE_MODEL_NAME}_MERA_Quant_Results"
RAW_LOGS_DIR = SAVE_DIR / "raw_logs"
CHECKPOINT_PATH = str(SAVE_DIR / "checkpoint.json")

MERA_REPO_DIR = Path("./MERA").resolve()
LM_EVAL_PATH = MERA_REPO_DIR / "lm-evaluation-harness"
MERA_TASKS_PATH = MERA_REPO_DIR / "benchmark_tasks"

LOCAL_MODEL_DIR = Path("./models").resolve()
LOCAL_BIN_DIR = Path("./llama_cpp_bin").resolve()
LOG_DIR = Path("./logs").resolve()
LOCAL_RESULTS_DIR = Path("./results").resolve()

for d in (SAVE_DIR, RAW_LOGS_DIR, MERA_REPO_DIR.parent, LOCAL_MODEL_DIR, LOCAL_BIN_DIR, LOG_DIR, LOCAL_RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

def get_time():
    return datetime.now().strftime("%H:%M:%S")

# ---------- Вывод конфигурации ----------
BACKEND_DESC = {
    "hf":                 "Путь A: прямая загрузка GGUF через transformers (сервер не нужен)",
    "gguf":               "Путь B: нативный бэкенд lm_eval → llama-server (teacher forcing, автопиннинг слотов)",
    "local-completions":  "Путь C: кастомный llama-server с PR #27537 (echo+logprobs)",
}
NEEDS_SERVER = BACKEND in ("gguf", "local-completions")

print("=" * 70)
print(f"🎯 Модель: {REPO_ID}")
print(f"⚙️  Бэкенд: {BACKEND}")
print(f"   → {BACKEND_DESC[BACKEND]}")
print(f"📋 Пресет «{PRESETS[PRESET]['title']}» — задач: {len(TASKS)}, LIMIT={LIMIT or '∞'}")
print(f"   {PRESETS[PRESET]['hint']}")
print(f"   {'─' * 60}")
for t in TASKS:
    info = TASK_INFO[t]
    kind = "loglikelihood" if info["type"] == "mc" else "генерация"
    status = "🔓" if info["open"] else "🔒"
    print(f"   {status} {t:15s} [{kind:14s}, {info['fewshot']}-shot, {info['metric']:12s}] — {info['desc']}")
print(f"   {'─' * 60}")
print(f"🎲 seed={SEED}, порт={SERVER_PORT}, VRAM-лимит={MAX_VRAM_GB} ГБ, FORCE_RERUN={FORCE_RERUN}")
print(f"💬 Системный промпт: {'задан' if SYSTEM_PROMPT else 'не задан (None)'}")
print(f"🦙 Сервер llama.cpp: {'нужен' if NEEDS_SERVER else 'НЕ нужен'}")
print(f"📦 llama.cpp: {LLAMA_CPP_ARCHIVE}")
print(f"📂 Результаты: {SAVE_DIR}")
n_open = sum(1 for t in TASKS if TASK_INFO[t]["open"])
n_mc = sum(1 for t in TASKS if TASK_INFO[t]["type"] == "mc")
n_gen = len(TASKS) - n_mc
print(f"📊 Открытых: {n_open}, loglikelihood: {n_mc}, генеративных: {n_gen}")
print("=" * 70)

In [ ]:
# @title БЛОК 4: УСТАНОВКА MERA (форк lm-evaluation-harness)
import importlib, importlib.metadata

os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "0")
os.environ.setdefault("HF_DATASETS_DISABLE_XET", "1")
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "120")

# 1. Клонирование MERA с подмодулями.
# Сабмодуль: artemorloff/lm-evaluation-harness @ feat/text_benches (lm-eval 0.4.13.dev0) —
# совместим с transformers 5.x из коробки, патчи и пины не нужны (подробнее — в README).
if not MERA_REPO_DIR.exists():
    print("📥 Клонирование MERA (с подмодулями)...")
    !git clone --recurse-submodules https://github.com/MERA-Evaluation/MERA.git
else:
    print("ℹ️ MERA уже склонирован. Обновление подмодулей...")
    !cd MERA && git pull --all --rebase --recurse-submodules
assert LM_EVAL_PATH.exists(), f"❌ {LM_EVAL_PATH} не существует — проверьте подмодули MERA."

# 2. Установка форка по официальной инструкции + extra [api] для бэкенда local-completions.
# База форка уже включает datasets/sqlitedict/dill/sacrebleu/rouge-score; extra [api]
# добавляет requests/aiohttp/tenacity/tqdm/tiktoken; transformers/torch/pandas/
# matplotlib/huggingface_hub предустановлены в Colab — отдельные списки зависимостей не нужны.
!pip install -e "{LM_EVAL_PATH}[api,hf]"
!pip install gguf
!apt-get install nvtop
# 3. Проверка установки. ВАЖНО: lm_eval в этом ноутбуке вызывается ТОЛЬКО через CLI
#    (subprocess в Блоке 6) — импорта в ядре не требуется: editable-установка (PEP 660)
#    видна только процессам, запущенным ПОСЛЕ pip install. Если установка не удалась,
#    следующая строка поднимет PackageNotFoundError.
importlib.invalidate_caches()
LM_EVAL_VERSION = importlib.metadata.version("lm_eval")
TRANSFORMERS_VERSION = importlib.metadata.version("transformers")
print(f"✅ Блок 4 завершён: lm_eval {LM_EVAL_VERSION}, transformers {TRANSFORMERS_VERSION}.")


In [ ]:
# @title БЛОК 5: РАЗВЁРТЫВАНИЕ LLAMA.CPP ИЗ АРХИВА (SHA-256 по manifest.json)
import hashlib

def sha256_file(path, chunk=8 * 1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

archive = Path(LLAMA_CPP_ARCHIVE)
manifest_path = Path(LLAMA_CPP_MANIFEST)
assert archive.exists(), f"❌ Архив не найден: {archive}"
assert manifest_path.exists(), f"❌ manifest.json не найден: {manifest_path}"

manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
art = manifest["артефакты"]["native/gpu_all"]
print(f"📦 Архив: {archive} ({archive.stat().st_size / 1e6:.0f} МБ)")
print(f"🔖 Сборка: {manifest['ветка']}, коммит {manifest['коммит']}, CUDA {manifest['cuda_архитектуры']['gpu_all']}")

actual_sha = sha256_file(archive)
if actual_sha.lower() != art["sha256_tar"].lower():
    raise RuntimeError(f"❌ SHA-256 архива не совпал с manifest.json!\n  ожидание: {art['sha256_tar']}\n  факт:     {actual_sha}")
print("✅ SHA-256 подтверждён.")

server_bin = next(LOCAL_BIN_DIR.rglob("llama-server"), None)
if server_bin is None:
    print("📥 Распаковка архива...")
    r = subprocess.run(f"tar -xzf '{archive}' -C '{LOCAL_BIN_DIR}'",
                       shell=True, capture_output=True, text=True, timeout=300)
    if r.returncode != 0:
        raise RuntimeError(f"❌ Ошибка распаковки: {r.stderr[-800:]}")
    server_bin = next(LOCAL_BIN_DIR.rglob("llama-server"), None)
assert server_bin is not None, "❌ llama-server не найден в архиве."
server_bin.chmod(0o755)

env = dict(os.environ)
#env["LD_LIBRARY_PATH"] = f"{server_bin.parent}:{env.get('LD_LIBRARY_PATH', '')}"
r = subprocess.run([str(server_bin), "--version"], capture_output=True, text=True,
                   env=env, timeout=30)
if r.returncode != 0:
    out = (r.stdout or "")[-800:] + (r.stderr or "")[-800:]
    raise RuntimeError(f"❌ llama-server --version завершился с кодом {r.returncode}: {out}")
ver_line = ((r.stdout or r.stderr or "(пусто)").strip().splitlines() or ["(пусто)"])[0]
print(f"✅ llama-server: {ver_line}")

ldd = subprocess.run(["ldd", str(server_bin)], capture_output=True, text=True,
                     env=env).stdout
missing = [l.strip() for l in ldd.splitlines() if "not found" in l]
if missing:
    raise RuntimeError(f"❌ Недостающие библиотеки: {missing}")

LLAMA_SERVER_BIN = str(server_bin)
LLAMA_BUILD_INFO = {"sha256_tar": actual_sha, "commit": manifest["коммит"],
                    "branch": manifest["ветка"], "cuda": manifest["cuda_архитектуры"]["gpu_all"]}
print(f"✅ Блок 5 завершён: {LLAMA_SERVER_BIN}")


In [ ]:
# @title БЛОК 6: ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
import requests
from huggingface_hub import HfApi, hf_hub_url, get_hf_file_metadata, hf_hub_download, snapshot_download

# ---------- 6.1 Каталог квантов и эталон ----------
def get_available_quants(repo_id):
    print(f"🔍 Кванты в {repo_id}...")
    quants = []
    for f in HfApi().list_repo_files(repo_id, repo_type="model"):
        if f.endswith(".gguf") and "mmproj" not in f.lower():
            meta = get_hf_file_metadata(hf_hub_url(repo_id, f, repo_type="model"))
            quants.append({"filename": f, "size_gb": meta.size / (1024 ** 3)})
    quants.sort(key=lambda x: x["size_gb"])
    print(f"✅ Найдено квантов: {len(quants)}")
    return quants

def select_reference_model(quants, max_vram_gb):
    for tag in ("BF16", "Q8_0"):
        cand = [q for q in quants if tag in q["filename"].upper()]
        if cand and cand[0]["size_gb"] + 4.0 <= max_vram_gb:
            return cand[0]
    fitting = [q for q in quants if q["size_gb"] + 4.0 <= max_vram_gb]
    return max(fitting, key=lambda x: x["size_gb"]) if fitting else None

def short_name(filename):
    return filename.replace(f"{BASE_MODEL_NAME}-", "").replace(".gguf", "")

# ---------- 6.2 Чекпоинт ----------
CONFIG_KEYS = ["repo_id", "preset", "limit", "seed", "tasks", "system_prompt"]
VERSION_KEYS = ["transformers_version", "lm_eval_version", "llama_cpp_sha256"]

def build_config_meta():
    return {"repo_id": REPO_ID, "preset": PRESET, "limit": LIMIT, "seed": SEED,
            "tasks": sorted(TASKS), "system_prompt": bool(SYSTEM_PROMPT),
            "transformers_version": TRANSFORMERS_VERSION,
            "lm_eval_version": LM_EVAL_VERSION,
            "llama_cpp_sha256": LLAMA_BUILD_INFO["sha256_tar"][:16]}

def fresh_state():
    return {"config_meta": build_config_meta(), "done": {}, "reference": None}

def load_checkpoint():
    if FORCE_RERUN:
        print("⚠️ FORCE_RERUN=True: чекпоинт игнорируется, всё пересчитывается.")
        return fresh_state()
    p = Path(CHECKPOINT_PATH)
    if p.exists():
        try:
            state = json.loads(p.read_text(encoding="utf-8"))
        except Exception as e:
            print(f"⚠️ Чекпоинт повреждён ({e}) — начинаем с чистого листа.")
            return fresh_state()
        meta = build_config_meta()
        old = state.get("config_meta", {})
        diff = [k for k in CONFIG_KEYS if old.get(k) != meta[k]]
        if diff:
            raise RuntimeError(
                f"❌ Чекпоинт {p} собран с другим конфигом (отличается: {diff}).\n"
                f"   Установите FORCE_RERUN=True или удалите {SAVE_DIR}.")
        for k in VERSION_KEYS:
            if old.get(k) != meta[k]:
                print(f"⚠️ Дрейф {k}: чекпоинт={old.get(k)} → сейчас={meta[k]}.")
        print(f"💾 Чекпоинт: готово квантов — {len(state['done'])}.")
        return state
    return fresh_state()

def save_checkpoint(state):
    tmp = CHECKPOINT_PATH + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(state, f, ensure_ascii=False, indent=1)
    os.replace(tmp, CHECKPOINT_PATH)

# ---------- 6.3 Токенизатор ----------
def get_tokenizer_path(repo_id):
    base_repo = repo_id.replace("-GGUF", "")
    safe = base_repo.replace("/", "_")
    local_dir = LOCAL_MODEL_DIR / f"tokenizer_{safe}"
    drive_dir = SAVE_DIR / "tokenizers" / f"tokenizer_{safe}"

    if not (local_dir.exists() and any(local_dir.iterdir())):
        if drive_dir.exists() and any(drive_dir.iterdir()):
            print(f"  📥 Токенизатор из кэша Drive...")
            shutil.copytree(drive_dir, local_dir, dirs_exist_ok=True)
        else:
            print(f"  ⬇️ Токенизатор {base_repo}...")
            snapshot_download(repo_id=base_repo, local_dir=str(local_dir),
                              allow_patterns=["tokenizer*", "special_tokens_map.json",
                                              "tokenizer_config.json", "vocab.json",
                                              "merges.txt", "*.model", "config.json"],
                              token=os.environ.get("HF_TOKEN"))

    cfg = local_dir / "config.json"
    if cfg.exists():
        try:
            config = json.loads(cfg.read_text(encoding="utf-8"))
            changed = False
            for key in ("routed_scaling_factor", "rope_theta", "sliding_window"):
                if key in config and isinstance(config[key], int):
                    config[key] = float(config[key])
                    changed = True
            if changed:
                cfg.write_text(json.dumps(config, indent=2), encoding="utf-8")
        except Exception as e:
            print(f"  ⚠️ Патч config.json не удался: {e}")

    if not (drive_dir.exists() and any(drive_dir.iterdir())):
        drive_dir.mkdir(parents=True, exist_ok=True)
        for item in local_dir.iterdir():
            if item.is_file():
                shutil.copy2(item, drive_dir / item.name)
    return str(local_dir)

# ---------- 6.4 Сервер llama.cpp ----------
def kill_existing_server(port):
    r = subprocess.run(f"lsof -t -i:{port}", shell=True, capture_output=True, text=True)
    for pid in (r.stdout.strip().split("\n") if r.stdout.strip() else []):
        if pid:
            subprocess.run(f"kill -9 {pid}", shell=True, capture_output=True)
            time.sleep(1)
    subprocess.run("pkill -9 -f llama-server", shell=True, capture_output=True)

def start_llama_server(model_path, port, alias):
    stdout_f = open(LOG_DIR / "server_stdout.log", "a", encoding="utf-8")
    stderr_f = open(LOG_DIR / "server_stderr.log", "a", encoding="utf-8")
    cmd = [LLAMA_SERVER_BIN, "-m", str(model_path),
           "--host", "127.0.0.1", "--port", str(port),
           "-ngl", "-1", "-c", "8192", "-b", "2048", "-t", "6",
           "-np", "4",
           "--seed", str(SEED), "--alias", alias,
           "-lv", "4",
           "--kv-unified",
           #"--ctx-checkpoints", "0",
           "--flash-attn", "auto",
           "--temp", "1.0", "--top-p", "0.95", "--top-k", "20", "--min-p", "0.00"]
    env = dict(os.environ)
    proc = subprocess.Popen(cmd, stdout=stdout_f, stderr=stderr_f, env=env,
                            start_new_session=True)
    for _ in range(60):
        if proc.poll() is not None:
            stop_llama_server(proc, stdout_f, stderr_f)
            err = (LOG_DIR / "server_stderr.log").read_text(encoding="utf-8")[-2000:]
            raise RuntimeError(f"❌ Сервер упал при старте (code {proc.returncode}).\n{err}")
        try:
            if requests.get(f"http://127.0.0.1:{port}/health", timeout=2).status_code == 200:
                return proc, stdout_f, stderr_f
        except Exception:
            pass
        time.sleep(2)
    stop_llama_server(proc, stdout_f, stderr_f)
    raise RuntimeError("❌ Сервер не ответил за 120 сек.")

def stop_llama_server(proc, stdout_f, stderr_f):
    if proc:
        try:
            os.killpg(os.getpgid(proc.pid), 15)
        except Exception:
            try: proc.terminate()
            except Exception: pass
        try: proc.wait(timeout=5)
        except Exception:
            try: os.killpg(os.getpgid(proc.pid), 9)
            except Exception: proc.kill()
    for f in (stdout_f, stderr_f):
        if f:
            try: f.close()
            except Exception: pass

def smoke_test_echo_logprobs(port):
    """Критический гейт: echo+logprobs в /v1/completions (PR #27537)."""
    try:
        r = requests.post(
            f"http://127.0.0.1:{port}/v1/completions",
            json={"prompt": "Столица России — Москва. Столица Франции — ",
                  "max_tokens": 1, "echo": True, "logprobs": 5},
            timeout=180)
        r.raise_for_status()
        lp = (r.json()["choices"][0].get("logprobs") or {}).get("token_logprobs")
        n = len(lp) if lp else 0
        if n < 5:
            return False, f"получено {n} logprob-ов (нужно ≥ 5)"
        return True, f"OK: {n} logprob-ов на echo-запросе"
    except Exception as e:
        return False, f"запрос не прошёл: {e!r}"

# ---------- 6.5 Разрешение имени метрики (универсальный резолвер) ----------
def resolve_metric(res, metric_name):
    """
    Ищет метрику в результатах lm_eval, перебирая все возможные суффиксы.
    MERA YAML-файлы используют filter_list name='scoring' → суффикс ',scoring'.
    Стандартный lm_eval использует ',none'. Также допускается отсутствие суффикса.
    """
    for suffix in [",scoring", ",none", ""]:
        key = metric_name + suffix
        if key in res and isinstance(res[key], (int, float)):
            return res[key]
    return None

# ---------- 6.5 Запуск одной задачи (три архитектурных пути) ----------
def run_task(quant_name, task, base_repo, tokenizer_path, model_path=None):
    """
    Запускает lm_eval для одной задачи.
    model_path: путь к GGUF-файлу (обязателен для бэкенда 'hf').
    """
    out_dir = LOCAL_RESULTS_DIR / quant_name / task
    out_dir.mkdir(parents=True, exist_ok=True)

    # ══════════════════════════════════════════════════════════════
    # Формирование model_args и cmd в зависимости от BACKEND
    # ══════════════════════════════════════════════════════════════

    if BACKEND == "hf":
        # ── Путь A: прямая загрузка GGUF через transformers ──────
        # model_args как в документации: pretrained=dir,gguf_file=file,tokenizer=path
        model_args = (f"pretrained={model_path.parent},"
                      f"gguf_file={model_path.name},"
                      f"tokenizer={tokenizer_path},"
                      f"dtype=bfloat16")
        cmd = ["lm_eval", "--model", "hf", "--model_args", model_args,
               "--tasks", task, "--include_path", str(MERA_TASKS_PATH),
               "--output_path", str(out_dir), "--log_samples",
               "--seed", str(SEED), "--device", "cuda:0",
               "--batch_size", "1", "--verbosity", "ERROR"]

    elif BACKEND == "gguf":
        # ── Путь B: нативный бэкенд gguf → llama-server ─────────
        # Использует teacher forcing через logit_bias, автопиннинг слотов
        model_args = f"base_url=http://127.0.0.1:{SERVER_PORT}"
        cmd = ["lm_eval", "--model", "gguf", "--model_args", model_args,
               "--tasks", task, "--include_path", str(MERA_TASKS_PATH),
               "--output_path", str(out_dir), "--log_samples",
               "--seed", str(SEED), "--batch_size", "1", "--verbosity", "ERROR"]

    else:  # BACKEND == "local-completions"
        # ── Путь C: кастомный бэкенд с echo+logprobs ─────────────
        # Требует llama-server с PR #27537
        model_args = (f"model={base_repo},"
                      f"base_url=http://127.0.0.1:{SERVER_PORT}/v1/completions,"
                      f"num_concurrent=8,"
                      f"tokenizer_backend=huggingface,"
                      f"tokenizer={tokenizer_path},"
                      f"tokenized_requests=False,"
                      f"timeout=1000000")
        cmd = ["lm_eval", "--model", "local-completions", "--model_args", model_args,
               "--tasks", task, "--include_path", str(MERA_TASKS_PATH),
               "--output_path", str(out_dir), "--log_samples",
               "--seed", str(SEED), "--batch_size", "1", "--verbosity", "ERROR"]

    # ── Общие параметры для всех бэкендов ─────────────────────────

    # 1. Few-shot
    fewshot = TASK_INFO.get(task, {}).get("fewshot", 0)
    cmd += ["--num_fewshot", str(fewshot)]

    # 2. Системный промпт
    if SYSTEM_PROMPT:
        cmd += ["--system_instruction", str(SYSTEM_PROMPT)]

    # 3. Лимит выборки
    if LIMIT:
        cmd += ["--limit", str(LIMIT)]

    # 4. Gen-Kwargs (как в официальном run_benchmark.sh)
    # Gen-kwargs: CLI НЕ передаём ни для одной задачи. По MODEL_SCORING.md
    # CLI --gen_kwargs полностью замещает YAML-параметры задачи, а официальный
    # run_benchmark.sh MERA его не использует. Действуют YAML-умолчания:
    #   - обычные генеративные задачи (базовый шаблон custom_generate_task.yaml):
    #     do_sample=false (жадное, детерминированное) + until=["\n"];
    #   - кодовые задачи (ruhumaneval/rucodeeval): do_sample=true, temperature=0.6
    #     + стопы "\nclass", "\ndef", "\n#", "\nif", "\nprint" (repeats=10
    #     требует сэмплирования, иначе pass@k вырождается).
    # Передача do_sample=False через CLI отключала официальный until=["\n"]
    # и загрязняла ответы «хвостом» до лимита токенов — особенно rudetox,\n    # где скоринг (sta/sim/fl) оценивает весь ответ целиком.

    # ── Запуск ────────────────────────────────────────────────────
    env = dict(os.environ)
    env["PYTHONPATH"] = str(MERA_REPO_DIR)
    env["TOKENIZERS_PARALLELISM"] = "false"
    env["HF_DATASETS_IN_MEMORY_MAX_SIZE"] = "23400000"
    env["HF_TOKEN"] = os.environ.get("HF_TOKEN", "")
    t0 = time.time()
    try:
        result = subprocess.run(cmd, env=env, capture_output=True, text=True, timeout=7200)
    except subprocess.TimeoutExpired:
        print(f"  ❌ [{get_time()}] {task}: таймаут CLI > 2 ч")
        return None
    wall_s = round(time.time() - t0, 1)

    task_log_dir = RAW_LOGS_DIR / quant_name / task
    task_log_dir.mkdir(parents=True, exist_ok=True)
    for f in out_dir.rglob("results_*.json"):
        shutil.copy2(f, task_log_dir / f.name)
    for f in out_dir.rglob(f"samples_{task}_*.jsonl"):
        shutil.copy2(f, task_log_dir / f.name)
    (task_log_dir / "task_meta.json").write_text(
        json.dumps({"quant": quant_name, "task": task, "wall_s": wall_s,
                    "backend": BACKEND, "exit_code": result.returncode, "cmd": cmd,
                    "stderr_tail": result.stderr[-2000:]}, ensure_ascii=False, indent=1),
        encoding="utf-8")
    parsed = parse_task_metrics(quant_name, task)
    if parsed is not None:
        parsed["wall_s"] = wall_s
    if result.returncode != 0 or parsed is None:
        src_log = LOG_DIR / "server_stderr.log"
        if NEEDS_SERVER and src_log.exists():
            shutil.copy2(src_log, task_log_dir / "server_stderr.log")
    if result.returncode != 0:
        print(f"  ❌ [{get_time()}] {task}: exit {result.returncode}: {result.stderr[-400:]}")
        return None
    return parsed

# ---------- 6.7 Парсинг метрик (универсальный, с поддержкой всех типов) ----------
def parse_task_metrics(quant_name, task):
    """
    Извлекает метрики из results_*.json.
    Поддерживает все типы метрик MERA:
      - acc (loglikelihood задачи)
      - exact_match (simplear, multiq, rumodar)
      - pass@1 (ruhumaneval, rucodeeval)
      - mcc (ruethics — 15 индивидуальных метрик)
      - grade_norm (use)
      - j (rudetox)
    """
    files = sorted((RAW_LOGS_DIR / quant_name / task).glob("results_*.json"),
                   key=lambda p: p.stat().st_mtime)
    if not files:
        return None
    raw = json.loads(files[-1].read_text(encoding="utf-8"))
    res = (raw.get("results") or {}).get(task)
    if not isinstance(res, dict):
        return None

    # Собираем все числовые метрики
    metrics = {k: v for k, v in res.items() if isinstance(v, (int, float))}
    metric_name = TASK_INFO[task]["metric"]

    # ── Специальная обработка: ruethics (15 MCC-метрик) ──────
    if metric_name == "mcc":
        mcc_keys = [k for k, v in res.items()
                    if k.startswith("mcc_") and isinstance(v, (int, float))]
        if mcc_keys:
            primary = round(sum(res[k] for k in mcc_keys) / len(mcc_keys), 4)
        else:
            primary = None
    # ── Специальная обработка: rudetox (J-метрика) ───────────
    elif metric_name == "j":
        primary = resolve_metric(res, "j")
        if primary is None:
            # Фоллбек: среднее из sta, sim, fl
            vals = [resolve_metric(res, m) for m in ("sta", "sim", "fl")]
            vals = [v for v in vals if v is not None]
            primary = round(sum(vals) / len(vals), 4) if vals else None
        else:
            primary = round(primary, 4)
    # ── Стандартные метрики: acc, exact_match, pass@1, grade_norm ─
    else:
        primary = resolve_metric(res, metric_name)
        primary = round(primary, 4) if isinstance(primary, (int, float)) else None

    return {"primary": primary, "metrics": metrics}

# ---------- 6.8 Скачивание, место, очистка ----------
def check_disk_space(required_gb):
    free_gb = shutil.disk_usage("/").free / (1024 ** 3)
    if free_gb < required_gb:
        raise RuntimeError(f"⚠️ Мало места: свободно {free_gb:.1f} ГБ, требуется {required_gb:.1f} ГБ.")

def download_model(filename):
    print(f"  ⬇️ Скачивание {filename}...")
    t0 = time.time()
    path = hf_hub_download(repo_id=REPO_ID, filename=filename,
                           local_dir=str(LOCAL_MODEL_DIR),
                           token=os.environ.get("HF_TOKEN"))
    size_gb = round(os.path.getsize(path) / (1024 ** 3), 2)
    print(f"  ✅ {size_gb} ГБ за {time.time() - t0:.0f} сек")
    return Path(path), size_gb

def cleanup(model_path):
    gc.collect()
    time.sleep(1)
    if model_path and os.path.exists(model_path):
        os.remove(model_path)
    hf_cache = Path.home() / ".cache" / "huggingface" / "hub"
    if hf_cache.exists():
        shutil.rmtree(hf_cache, ignore_errors=True)
    print("  🧹 Очистка выполнена.")

# ---------- 6.9 Валидация ----------
_required = ["get_available_quants", "select_reference_model", "short_name",
             "load_checkpoint", "save_checkpoint", "get_tokenizer_path",
             "kill_existing_server", "start_llama_server", "stop_llama_server",
             "smoke_test_echo_logprobs", "resolve_metric", "run_task",
             "parse_task_metrics", "check_disk_space", "download_model", "cleanup"]
_missing = [n for n in _required if n not in globals()]
assert not _missing, f"❌ Не определены функции: {_missing}"
print(f"✅ Блок 6 завершён: {len(_required)} функций определены.")
print(f"⚙️  Бэкенд: {BACKEND} ({BACKEND_DESC[BACKEND]})")

In [ ]:
# @title БЛОК 7: ОСНОВНОЙ ЦИКЛ (ЭТАП 0 — эталон, ЭТАП 1 — кванты)
state = load_checkpoint()
done = state["done"]

all_quants = get_available_quants(REPO_ID)
ref_q = select_reference_model(all_quants, MAX_VRAM_GB)
if ref_q is None:
    raise RuntimeError(f"❌ Ни один квант не влезает в {MAX_VRAM_GB} ГБ VRAM.")
REF_NAME = short_name(ref_q["filename"])
print(f"🏆 Эталон: {ref_q['filename']} ({ref_q['size_gb']:.1f} ГБ)")

def evaluate_quant(q_info, is_reference=False):
    name = short_name(q_info["filename"])
    pending = [t for t in TASKS if not (name in done and t in done.get(name, {}) and t != "size_gb")]
    pending = [t for t in pending if t in TASK_INFO]
    if name in done and not pending:
        print(f"⏭️ [{get_time()}] {name}: все задачи готовы (чекпоинт).")
        return
    print(f"\n{'=' * 70}\n▶ [{get_time()}] {name} ({q_info['size_gb']:.1f} ГБ), задач: {len(pending)}\n{'=' * 70}")
    check_disk_space(q_info["size_gb"] * 2.2 + 5.0)
    model_path = None
    server = None
    try:
        model_path, _size = download_model(q_info["filename"])
        tokenizer_path = get_tokenizer_path(REPO_ID)

        # ── Условный запуск сервера ──────────────────────────────
        if NEEDS_SERVER:
            kill_existing_server(SERVER_PORT)
            print(f"  🚀 [{get_time()}] Запуск llama-server (бэкенд={BACKEND}, seed={SEED})...")
            server = start_llama_server(model_path, SERVER_PORT, alias=REPO_ID.replace("-GGUF", ""))
            if is_reference and BACKEND == "local-completions":
                ok, msg = smoke_test_echo_logprobs(SERVER_PORT)
                print(f"  🧪 Гейт echo/logprobs: {msg}")
                if not ok:
                    raise RuntimeError(
                        "❌ Сборка llama-server не отдаёт echo+logprobs — loglikelihood-задачи "
                        "не пройдут. Проверьте архив (PR #27537) и Блок 5.")
        else:
            print(f"  📦 [{get_time()}] Бэкенд '{BACKEND}': сервер не нужен, модель загружается напрямую.")

        row = done.setdefault(name, {})
        row["size_gb"] = q_info["size_gb"]
        row["backend"] = BACKEND
        if is_reference:
            row["is_reference"] = True
            state["reference"] = name
        for task in pending:
            print(f"  🧪 [{get_time()}] {task} ... ", end="", flush=True)
            # Для бэкенда hf передаём model_path, для остальных — None
            res = run_task(name, task, REPO_ID.replace("-GGUF", ""), tokenizer_path,
                          model_path=model_path)
            if res is None:
                print("❌ (не в чекпоинт — повторится при следующем запуске)")
                continue
            row[task] = res
            save_checkpoint(state)
            print(f"✅ primary={res['primary']} ({res['wall_s']} c)")
        save_checkpoint(state)
    except KeyboardInterrupt:
        print(f"\n⏹️ [{get_time()}] {name}: прервано. Выполненное — в чекпоинте.")
        raise
    finally:
        if server:
            stop_llama_server(*server)
            kill_existing_server(SERVER_PORT)
        cleanup(model_path)

# ---------- ЭТАП 0: эталон (BF16 → Q8_0) ----------
if REF_NAME in done and all(t in done[REF_NAME] for t in TASKS):
    print(f"\n⏭️ ЭТАП 0: эталон {REF_NAME} уже посчитан (чекпоинт).")
else:
    print(f"\n{'=' * 70}\n🏁 ЭТАП 0: ЭТАЛОН {ref_q['filename']}\n{'=' * 70}")
    evaluate_quant(ref_q, is_reference=True)

# ---------- ЭТАП 1: остальные кванты ----------
print(f"\n{'=' * 70}\n🔄 ЭТАП 1: ОСТАЛЬНЫЕ КВАНТЫ\n{'=' * 70}")
for q in all_quants:
    name = short_name(q["filename"])
    if name == REF_NAME:
        continue
    if q["size_gb"] + 4.0 > MAX_VRAM_GB:
        print(f"⏭️ {name}: не влезает в VRAM ({q['size_gb']:.1f} ГБ + 4 ГБ > {MAX_VRAM_GB} ГБ)")
        continue
    evaluate_quant(q)

print(f"\n🏁 [{get_time()}] ЦИКЛ ЗАВЕРШЁН. Переходите к Блоку 8.")

In [ ]:
# @title БЛОК 8: АГРЕГАЦИЯ, ДЕГРАДАЦИЯ, ГРАФИКИ, ОТЧЁТ
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

state = json.loads(Path(CHECKPOINT_PATH).read_text(encoding="utf-8"))
done = state["done"]
meta = state.get("config_meta", {})

rows = []
for name, cell in done.items():
    row = {"Quant": name, "Size_GB": cell.get("size_gb"),
           "Is_Ref": bool(cell.get("is_reference"))}
    for t in TASKS:
        row[t] = (cell.get(t) or {}).get("primary")
    rows.append(row)
df = pd.DataFrame(rows).sort_values("Size_GB").reset_index(drop=True)

ref_rows = df[df["Is_Ref"]]
ref_row = ref_rows.iloc[0] if len(ref_rows) else None

# Правило ненадёжности: у эталона метрика None/NaN/0 → задача исключается из средних
unreliable = []
for t in TASKS:
    v = ref_row.get(t) if ref_row is not None else None
    if v is None or (isinstance(v, float) and (np.isnan(v) or v == 0)):
        unreliable.append(t)
reliable = [t for t in TASKS if t not in unreliable]
if unreliable:
    print(f"⚠️ Ненадёжные задачи (у эталона 0/нет данных), исключены из средних: {unreliable}")

df["AvgScore"] = df[reliable].mean(axis=1) if reliable else np.nan
ref_avg = float(ref_row["AvgScore"]) if ref_row is not None and not pd.isna(ref_row.get("AvgScore", np.nan)) else np.nan
df["Delta_pp"] = (df["AvgScore"] - ref_avg) * 100
df["Delta_pct"] = (df["AvgScore"] / ref_avg - 1) * 100 if ref_avg else np.nan
for t in reliable:
    df[f"{t}_dpp"] = (df[t] - float(ref_row[t])) * 100

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)
cols = ["Quant", "Size_GB", "Is_Ref"] + reliable + ["AvgScore", "Delta_pp", "Delta_pct"]
display(df[[c for c in cols if c in df.columns]].round(4))

df.to_csv(SAVE_DIR / "mera_results.csv", index=False, encoding="utf-8")
df[["Quant", "Size_GB", "Is_Ref", "AvgScore", "Delta_pp", "Delta_pct"]].round(4)     .to_csv(SAVE_DIR / "degradation.csv", index=False, encoding="utf-8")

# ---------- Графики ----------
sns_style = plt.style.context("default")
fig, axes = plt.subplots(1 + (len(reliable) + 3) // 4, 4, figsize=(20, 4.5 * (1 + (len(reliable) + 3) // 4)))
axes = np.atleast_2d(axes).ravel()
panels = reliable + ["AvgScore"]
for ax, t in zip(axes, panels):
    sub = df.dropna(subset=[t]) if t in df.columns else df.dropna(subset=["AvgScore"])
    ax.plot(sub["Size_GB"], sub[t], "o-")
    for _, r in sub.iterrows():
        ax.annotate(r["Quant"], (r["Size_GB"], r[t]), fontsize=7,
                    xytext=(0, 5), textcoords="offset points", ha="center")
    ax.set_xlabel("Размер, ГБ")
    ax.set_title(t + (" (среднее)" if t == "AvgScore" else ""), fontweight="bold")
    if t != "AvgScore" and ref_row is not None and not pd.isna(ref_row.get(t, np.nan)):
        ax.axhline(float(ref_row[t]), color="gray", ls=":", lw=1)
for ax in axes[len(panels):]:
    ax.axis("off")
fig.suptitle(f"Деградация метрик MERA: {REPO_ID} (пресет {PRESET}, LIMIT={LIMIT})", fontweight="bold")
fig.tight_layout()
fig.savefig(SAVE_DIR / "degradation_plots.png", dpi=200, bbox_inches="tight")
plt.show()

# ---------- Heatmap Δ п.п. ----------
if reliable:
    hm = df.set_index("Quant")[[f"{t}_dpp" for t in reliable]].round(2)
    fig2, ax2 = plt.subplots(figsize=(1.6 * len(reliable) + 4, 0.5 * len(hm) + 2))
    im = ax2.imshow(hm.values, cmap="RdYlGn", aspect="auto")
    ax2.set_xticks(range(len(hm.columns)), hm.columns, rotation=30, ha="right")
    ax2.set_yticks(range(len(hm.index)), hm.index)
    for i in range(hm.shape[0]):
        for j in range(hm.shape[1]):
            v = hm.values[i, j]
            if not np.isnan(v):
                ax2.text(j, i, f"{v:+.1f}", ha="center", va="center", fontsize=8)
    fig2.colorbar(im, label="Δ к эталону, п.п.")
    ax2.set_title("Деградация по задачам (п.п., меньше — хуже)", fontweight="bold")
    fig2.tight_layout()
    fig2.savefig(SAVE_DIR / "degradation_heatmap.png", dpi=200, bbox_inches="tight")
    plt.show()

# ---------- MD-отчёт ----------
def _md_table(dframe, floatfmt=4):
    d = dframe.round(floatfmt)
    head = "| " + " | ".join(d.columns) + " |"
    sep = "|" + "---|" * len(d.columns)
    body = "\n".join("| " + " | ".join(str(v) for v in row) + " |" for row in d.astype(object).values)
    return "\n".join([head, sep, body])

md = []
md.append(f"# Деградация MERA по квантам: {REPO_ID}")
md.append(f"**Пресет:** {PRESET} ({PRESETS[PRESET]['title']}), LIMIT={LIMIT}, seed={SEED}")
md.append(f"**Эталон:** {REF_NAME}")
md.append(f"**Среда:** transformers {meta.get('transformers_version')} · lm-eval {meta.get('lm_eval_version')} · "
          f"llama.cpp {LLAMA_BUILD_INFO['branch']}@{LLAMA_BUILD_INFO['commit']} (CUDA {LLAMA_BUILD_INFO['cuda']})")
md.append(f"**GPU:** {torch.cuda.get_device_properties(0).name}")
if unreliable:
    md.append(f"\n⚠️ **Ненадёжные задачи** (у эталона 0/нет данных — исключены из средних): {', '.join(unreliable)}")
md.append("\n## Сводная таблица\n")
md.append(_md_table(df[[c for c in cols if c in df.columns]]))
md.append("\n## Деградация по задачам (Δ п.п.)\n")
md.append(_md_table(df[["Quant", "Size_GB"] + [f"{t}_dpp" for t in reliable]]) if reliable else "Нет надёжных задач.")
(SAVE_DIR / "mera_quant_report.md").write_text("\n".join(md), encoding="utf-8")
print(f"\n💾 Сохранено в {SAVE_DIR}: mera_results.csv, degradation.csv, degradation_plots.png, "
      f"degradation_heatmap.png, mera_quant_report.md")


In [ ]:
# @title БЛОК 9: ФИНАЛЬНЫЙ РЕЙТИНГ (вердикты + Pareto)
def verdict(delta_pct):
    if delta_pct is None or (isinstance(delta_pct, float) and np.isnan(delta_pct)):
        return "нет данных"
    deg = -delta_pct
    if deg <= 1:  return "≈ без потерь"
    if deg <= 3:  return "рекомендуется"
    if deg <= 7:  return "приемлемо"
    return "⚠️ заметная деградация"

df["Verdict"] = df["Delta_pct"].apply(verdict)
df.loc[df["Is_Ref"], "Verdict"] = "🏆 ЭТАЛОН"
rank_cols = ["Quant", "Size_GB", "AvgScore", "Delta_pp", "Delta_pct", "Verdict"]
display(df[rank_cols].round(4))
df[rank_cols].round(4).to_csv(SAVE_DIR / "final_ranking.csv", index=False, encoding="utf-8")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
sub = df.dropna(subset=["AvgScore"])
ax1.plot(sub["Size_GB"], sub["AvgScore"], "o-")
for _, r in sub.iterrows():
    ax1.annotate(r["Quant"], (r["Size_GB"], r["AvgScore"]), fontsize=8,
                 xytext=(0, 6), textcoords="offset points", ha="center")
ax1.set_xlabel("Размер, ГБ"); ax1.set_ylabel("Средний балл MERA")
ax1.set_title(f"Средний балл vs размер · {BASE_MODEL_NAME}", fontweight="bold")

pts = sub[~sub["Is_Ref"]].sort_values(["AvgScore", "Size_GB"], ascending=[False, True])
front, best = [], np.inf
for _, r in pts.iterrows():
    if r["Size_GB"] < best:
        front.append(r["Quant"]); best = r["Size_GB"]
ax2.scatter(pts["Size_GB"], pts["AvgScore"], s=40, color="lightgray", label="Все кванты")
fp = pts[pts["Quant"].isin(front)]
ax2.scatter(fp["Size_GB"], fp["AvgScore"], s=80, color="tab:blue", label="Pareto-фронт")
for _, r in fp.iterrows():
    ax2.annotate(r["Quant"], (r["Size_GB"], r["AvgScore"]), fontsize=8,
                 xytext=(4, 4), textcoords="offset points")
ax2.set_xlabel("Размер, ГБ"); ax2.set_ylabel("Средний балл")
ax2.set_title("Pareto: меньше ГБ при том же качестве", fontweight="bold")
ax2.legend(fontsize=8)
fig.tight_layout()
fig.savefig(SAVE_DIR / "final_analysis.png", dpi=200, bbox_inches="tight")
plt.show()
print(f"💾 Итоговый рейтинг: {SAVE_DIR / 'final_ranking.csv'}")


In [ ]:
# @title БЛОК 10: ПЕРЕСБОРКА ИЗ СЫРЫХ ЛОГОВ (без GPU и без моделей)
rows = []
for quant_dir in sorted(p for p in RAW_LOGS_DIR.iterdir() if p.is_dir()):
    row = {"Quant": quant_dir.name}
    for task_dir in sorted(p for p in quant_dir.iterdir() if p.is_dir()):
        if task_dir.name not in TASK_INFO:
            continue
        parsed = parse_task_metrics(quant_dir.name, task_dir.name)
        if parsed:
            row[task_dir.name] = parsed["primary"]
            row[f"{task_dir.name}_wall_s"] = parsed.get("wall_s")
    rows.append(row)
df_re = pd.DataFrame(rows)
display(df_re)
df_re.to_csv(SAVE_DIR / "reparsed_from_raw.csv", index=False, encoding="utf-8")
print(f"💾 {SAVE_DIR / 'reparsed_from_raw.csv'}")


In [ ]:
# @title БЛОК 11: УПАКОВКА ДЛЯ САБМИТА (MERA Submission)
import sys
import os
import yaml

# ── Патч недостающих импортов в lm_eval.utils ──────────────────
# log_to_submission.py ожидает:
#   from lm_eval.utils import load_yaml_config, sanitize_model_name
# В форке 0.4.13.dev0 эти функции могут быть в других модулях.
# Подкладываем их в lm_eval.utils ДО импорта скрипта.

import lm_eval.utils

# 1. load_yaml_config
if not hasattr(lm_eval.utils, 'load_yaml_config'):
    try:
        from lm_eval.api.task import load_yaml_config as _lyc
    except ImportError:
        def _lyc(yaml_path):
            """Fallback: читаем YAML вручную."""
            with open(yaml_path, 'rb') as f:
                return yaml.full_load(f)
    lm_eval.utils.load_yaml_config = _lyc

# 2. sanitize_model_name
if not hasattr(lm_eval.utils, 'sanitize_model_name'):
    try:
        from lm_eval.loggers.evaluation_tracker import sanitize_model_name as _smn
    except ImportError:
        def _smn(model_name: str) -> str:
            """Fallback: заменяем спецсимволы для имени директории."""
            return model_name.replace("/", "_").replace(" ", "_").replace(":", "_")
    lm_eval.utils.sanitize_model_name = _smn

# ── Теперь можно безопасно импортировать ────────────────────────
original_cwd = os.getcwd()
os.chdir(str(MERA_REPO_DIR))

try:
    sys.path.insert(0, str(MERA_REPO_DIR / "scripts"))
    # Перезагружаем модуль, если он уже был импортирован
    if 'log_to_submission' in sys.modules:
        import importlib
        importlib.reload(sys.modules['log_to_submission'])
    from log_to_submission import create_submission

    def pack_for_submission(target_quant=None):
        """
        Упаковывает сырые логи в формат MERA для сабмита.
        Если target_quant=None, упаковывает логи эталона (reference).
        """
        state = json.loads(Path(CHECKPOINT_PATH).read_text(encoding="utf-8"))
        ref_name = state.get("reference")
        quant_to_pack = target_quant or ref_name

        if not quant_to_pack or quant_to_pack not in state.get("done", {}):
            print(f"❌ Квант {quant_to_pack} не найден в чекпоинте.")
            return

        print(f"📦 Упаковка логов для кванта: {quant_to_pack}")
        temp_outputs_dir = LOCAL_RESULTS_DIR / "temp_submission_logs"
        if temp_outputs_dir.exists():
            shutil.rmtree(temp_outputs_dir)
        temp_outputs_dir.mkdir(parents=True, exist_ok=True)

        # Копируем все samples и results файлы из RAW_LOGS_DIR в единую папку
        quant_logs_dir = RAW_LOGS_DIR / quant_to_pack
        if not quant_logs_dir.exists():
            print(f"❌ Папка с логами {quant_logs_dir} не найдена.")
            return

        for task_dir in quant_logs_dir.iterdir():
            if task_dir.is_dir():
                for f in task_dir.glob("*.json*"):
                    shutil.copy2(f, temp_outputs_dir / f.name)

        dst_dir = SAVE_DIR / f"submission_{quant_to_pack}"
        if dst_dir.exists():
            shutil.rmtree(dst_dir)

        try:
            # gen=False, так как MERA использует loglikelihood для большинства задач,
            # а генеративные задачи скрипт определяет сам по GENERATIVE_TASKS.
            create_submission(str(temp_outputs_dir.resolve()), str(dst_dir.resolve()), gen=False)
            print(f"✅ Сабмит успешно создан: {dst_dir}.zip")
        except Exception as e:
            print(f"❌ Ошибка при упаковке сабмита: {e}")
            import traceback
            traceback.print_exc()
        finally:
            shutil.rmtree(temp_outputs_dir, ignore_errors=True)

    # Запуск упаковки для эталонной модели
    pack_for_submission()

finally:
    os.chdir(original_cwd)